# Financial Report Assistant — Minh hoạ Workflow End-to-End

Notebook này minh hoạ **5 luồng chạy thực tế** của codebase, bằng cách gọi
trực tiếp các hàm thật trong `src/` và `main.py` — không mô phỏng, không
mock. Mọi cell code trong notebook này **chạy thật**: gọi SEC EDGAR thật,
tính embedding thật, gọi Gemini API thật.

## 5 luồng sẽ được minh hoạ (mỗi luồng một phần riêng, không gộp)

| # | Luồng | Kích hoạt bởi | Mục đích |
|---|---|---|---|
| A | **Index một filing** | `python main.py --ticker X --filing Y` | Tải, parse, chunk, embed, lưu 1 filing mới |
| B | **Force-rebuild** | `python main.py --force-rebuild` | Dựng lại toàn bộ FAISS index từ Markdown đã parse + manifest |
| C | **Truy vấn (Query)** | Streamlit chat / gọi trực tiếp | Hỏi đáp trên dữ liệu đã index, có trích dẫn |
| D | **Index từ giao diện** | Nút "Index a new filing" trên sidebar | Giống luồng A nhưng kích hoạt từ UI, có cache phải xử lý |
| E | **Đánh giá (Evaluation)** | `python -m src.evaluation.metrics` | Đo chất lượng retrieval trên bộ câu hỏi mẫu (golden set) |

## Quy ước trình bày

Mỗi hàm "đầu dòng" (quan trọng, là một bước rõ ràng trong luồng) được giới
thiệu theo đúng 4 phần:

1. **Tên hàm + Input** — ý nghĩa từng tham số; nếu tham số là dataclass thì
   giới thiệu chi tiết dataclass đó trước.
2. **Output** — kiểu dữ liệu trả về, ý nghĩa.
3. **Chức năng** — hàm làm gì.
4. **Tech stack** — công nghệ/thư viện hàm dùng, và vai trò của nó.

Nếu hàm A nhận input là output của hàm B, hàm B luôn được giới thiệu **trước**
hàm A. Trong mỗi luồng, các **bên tham gia** (User, CLI, SEC EDGAR, FAISS,
Gemini, File System...) được ghi rõ bằng mũi tên `Bên X → Bên Y`.

Các hàm phụ trợ nhỏ (private, một dòng) được nhắc gọn trong bullet "Tech
stack" của hàm cha thay vì tách section riêng. Các hàm không thuộc luồng nào
được gom vào **Phần F** ở cuối. Hàm test được bỏ qua hoàn toàn.

## Dữ liệu dùng để minh hoạ

Trước khi chạy notebook này, index đã có sẵn **NVDA 10-K** và **NVDA 10-Q**
(từ các lần chạy `main.py` trước đó). Notebook sẽ:
- Dùng **MSFT 10-K** để đi **thủ công từng hàm một** ở Phần A (gọi trực tiếp
  từng hàm, không qua `index_filing`) — mục đích là để thấy rõ input/output
  thật của từng bước.
- Dùng **GOOGL 10-K** để minh hoạ `index_filing()` chạy **trọn gói trong 1
  lệnh** (đối lập với cách làm thủ công ở trên).
- Sau đó cả 4 filing (NVDA 10-K, NVDA 10-Q, MSFT 10-K, GOOGL 10-K) đều nằm
  trong index chung — đúng với thiết kế "one shared FAISS index for all
  filings, distinguished only by metadata".

In [1]:
# --- Bootstrap: đưa thư mục gốc repo vào sys.path để import được `src.*` và `main` ---
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print("Repo root:", REPO_ROOT)
print("sys.path[0]:", sys.path[0])

Repo root: C:\Python\Personal Project\Financial Report Assistant
sys.path[0]: C:\Python\Personal Project\Financial Report Assistant


In [2]:
import json
import time
from dataclasses import asdict, fields
from datetime import datetime, timezone
from pprint import pprint

# Xác nhận .env đã được load qua src.utils.config (không in giá trị bí mật)
from src.utils.config import GEMINI_API_KEY, GEMINI_MODEL, SEC_USER_AGENT

print("GEMINI_API_KEY đã cấu hình:", bool(GEMINI_API_KEY))
print("GEMINI_MODEL:", GEMINI_MODEL)
print("SEC_USER_AGENT:", SEC_USER_AGENT)

GEMINI_API_KEY đã cấu hình: True
GEMINI_MODEL: gemini-flash-latest
SEC_USER_AGENT: Kiet Nguyen quanghuykiet05@gmail.com


---
# PHẦN A — Luồng Index một filing

**Kích hoạt bởi:** `python main.py --ticker MSFT --filing 10-K`

## Các bên tham gia

```
User (terminal)
   │  gõ lệnh CLI
   ▼
main.py (CLI)  ──────────────►  SEC EDGAR (data.sec.gov, www.sec.gov)
   │                                  │ tải file HTML filing
   │  ◄───────────────────────────────┘
   ▼
File System (data/raw/*.html, data/processed/*.md)
   │
   ▼
FAISS Vector Store (data/vectorstore/index.faiss, index.pkl)
   │
   ▼
Manifest (data/manifest.json)
```

## Cách trình bày trong Phần A

Phần A được chia làm 2 bước lớn:

1. **Đi thủ công từng hàm một** (dùng **MSFT**) — gọi trực tiếp từng hàm con
   theo đúng thứ tự phụ thuộc, để thấy rõ input/output thật của từng bước.
2. **Giới thiệu `index_filing()`** — hàm điều phối duy nhất gói toàn bộ các
   bước ở (1) lại làm một, được dùng bởi cả CLI (`main.py:main()`) và
   form trên Streamlit sidebar (Phần D) — **chỉ có một luồng code index duy
   nhất trong toàn bộ dự án**, không có bản sao thứ hai.

## A.1 — `get_embedding_model()`

**1. Tên hàm + Input**
- `get_embedding_model()` — **không có tham số**.

**2. Output**
- Trả về `HuggingFaceEmbeddings` (lớp của `langchain_huggingface`) — một
  đối tượng embedding model đã sẵn sàng dùng (đã load trọng số model vào bộ
  nhớ).

**3. Chức năng**
- Khởi tạo model embedding dùng chung cho toàn bộ dự án:
  `BAAI/bge-small-en-v1.5`, với `encode_kwargs={"normalize_embeddings": True}`
  (vector output luôn được chuẩn hoá độ dài 1, phù hợp cho so khớp bằng tích
  vô hướng/cosine).
- Đây là **điểm khởi tạo embedding duy nhất** trong toàn luồng — object trả
  về sau đó được truyền tiếp (dependency injection) xuống mọi hàm cần nó,
  thay vì mỗi hàm tự tạo model riêng (rất tốn — load model mất nhiều giây).

**4. Tech stack**
- **`langchain_huggingface`**: là một thư viện của Langchain, đóng vai trò như một plugin/adapter kết nối giữa Langchain và HuggingFace
- **`langchain_huggingface.HuggingFaceEmbeddings`**: 
  + Được kế thừa abstract class `Embeddings` trong `langchain_core`, là wrapper của LangChain quanh `sentence-transformers`, cho một API embedding thống nhất (`embed_query`, `embed_documents`) mà FAISS/retriever phía sau dùng được trực tiếp. 
  
- Lợi ích của wrapper: 
  + Sau này, nếu muốn đổi sang `sentence-tranformers`(embedding model) khác, chỉ cần đổi `model_name` thành những `sentence-tranformers` available trên HuggingFace Hub. 
  + Kể cả khi muốn đổi thành những embedding model thuộc provider khác, không phải của `sentence-transformer`, thì ta chỉ cần đổi wrapper class, miễn là wrapper class này cũng implement interface `Embeddings` (có embed_documents/embed_query)
- **`sentence-transformers`** (bên dưới `HuggingFaceEmbeddings`): tải và chạy
  model `BAAI/bge-small-en-v1.5` (384 chiều) trên CPU/GPU cục bộ — không gọi
  API ngoài, không tốn phí, nhưng cần tải model về máy lần đầu.

In [3]:
from src.embeddings.embedder import get_embedding_model

t0 = time.time()
embedding = get_embedding_model()
print(f"Loại đối tượng: {type(embedding).__name__}")
print(f"Model name: {embedding.model_name}")
print(f"Thời gian load: {time.time() - t0:.1f}s")

2026-08-21 19:28:49 | INFO     | src.embeddings.embedder | Loading embedding model: BAAI/bge-small-en-v1.5


Loại đối tượng: HuggingFaceEmbeddings
Model name: BAAI/bge-small-en-v1.5
Thời gian load: 9.8s


## A.2 — Dataclass `FilingRef` (giới thiệu trước vì `load_filing()` trả về nó)

`FilingRef` (định nghĩa trong `src/ingestion/sec_loader.py`) là một
`@dataclass` mô tả một filing đã được xác định + tải về:

| Field | Kiểu | Ý nghĩa |
|---|---|---|
| `doc_id` | `str` | Khoá định danh duy nhất, dạng `"{TICKER}_{filing_type}_{filing_date}"`, vd `"MSFT_10-K_2025-06-30"` |
| `ticker` | `str` | Mã cổ phiếu, đã viết hoa |
| `filing_type` | `str` | Loại form SEC chính xác, vd `"10-K"` |
| `filing_date` | `str` | Ngày nộp filing, ISO `YYYY-MM-DD` |
| `filing_name` | `str` | Tên hiển thị, vd `"MICROSOFT CORP 10-K 2025"` |
| `url` | `str` | URL tài liệu HTML gốc trên `www.sec.gov/Archives/...` |
| `source` | `str` | Tên file HTML, dạng `"{doc_id}.html"` |
| `cik` | `str \| None` | Central Index Key của công ty trên SEC (10 chữ số, đệm 0) |
| `accession` | `str \| None` | Số hồ sơ nộp SEC, dạng `"0000000000-00-000000"` |
| `raw_path` | `Path` | Đường dẫn file HTML đã tải về, `data/raw/{doc_id}.html` |

`doc_id`, `url`, `filing_name` là các trường **không thể suy ra được** từ nội
dung HTML/Markdown về sau — đây chính là lý do `data/manifest.json` cần tồn
tại (xem Phần B).

## A.3 — `load_filing(ticker: str, filing_type: str) -> FilingRef`

**1. Tên hàm + Input**
- `ticker: str` — mã cổ phiếu, vd `"MSFT"` (hàm tự viết hoa nếu chưa hoa).
- `filing_type: str` — form SEC chính xác, vd `"10-K"`, `"10-Q"`.

**2. Output**
- Trả về `FilingRef` (xem A.2) — mô tả đầy đủ filing tìm được, đã tải HTML
  về `data/raw/{doc_id}.html`.

**3. Chức năng**
- Resolve `ticker` → CIK (qua `company_tickers.json`, có cache cục bộ tại
  `data/.cache/company_tickers.json`).
- Gọi Submissions API của SEC để lấy toàn bộ lịch sử nộp hồ sơ, chọn filing
  **mới nhất** khớp đúng `filing_type`.
- **`doc_id` được tạo lần đầu tiên bên trong hàm này** (qua helper
  `build_doc_id`), ngay sau khi biết được `filing_date` thật từ SEC.
- Tải file HTML chính (`primaryDocument`) về `data/raw/{doc_id}.html` —
  **nếu file đã tồn tại thì tái sử dụng, không tải lại** (cache short-circuit
  theo đường dẫn file, độc lập với việc `doc_id` có nằm trong manifest/FAISS
  hay không).

**4. Tech stack**
- **`requests`** (qua helper `http_get`, có tự động retry 3 lần với backoff
  cho các mã lỗi 429/500/502/503/504): gọi 2 API JSON công khai của SEC
  EDGAR — `company_tickers.json` và `data.sec.gov/submissions/CIK{cik}.json`
  — và tải file HTML filing.
- **`BeautifulSoup` (lxml)**: chỉ dùng trong nhánh dự phòng hiếm gặp, khi
  Submissions API không trả `primaryDocument` (filing rất cũ) — khi đó hàm
  tự parse trang chỉ mục (index page) chính thức của SEC để tìm tên file
  HTML, **không bao giờ scrape trang tìm kiếm full-text của EDGAR**.

In [4]:
from src.ingestion.sec_loader import load_filing

ref_msft = load_filing("MSFT", "10-K")
print("doc_id      :", ref_msft.doc_id)
print("filing_name :", ref_msft.filing_name)
print("filing_date :", ref_msft.filing_date)
print("url         :", ref_msft.url)
print("cik         :", ref_msft.cik)
print("accession   :", ref_msft.accession)
print("raw_path    :", ref_msft.raw_path)
print("file tồn tại:", ref_msft.raw_path.exists())

2026-08-21 19:28:59 | INFO     | src.utils.helpers | GET https://data.sec.gov/submissions/CIK0000789019.json


2026-08-21 19:28:59 | INFO     | src.ingestion.sec_loader | Downloading filing doc_id=MSFT_10-K_2026-07-29 from https://www.sec.gov/Archives/edgar/data/789019/000119312526323660/msft-20260630.htm


2026-08-21 19:28:59 | INFO     | src.utils.helpers | GET https://www.sec.gov/Archives/edgar/data/789019/000119312526323660/msft-20260630.htm


doc_id      : MSFT_10-K_2026-07-29
filing_name : MICROSOFT CORP 10-K 2026
filing_date : 2026-07-29
url         : https://www.sec.gov/Archives/edgar/data/789019/000119312526323660/msft-20260630.htm
cik         : 0000789019
accession   : 0001193125-26-323660
raw_path    : C:\Python\Personal Project\Financial Report Assistant\data\raw\MSFT_10-K_2026-07-29.html
file tồn tại: True


## A.4 — `load_vectorstore(embedding: HuggingFaceEmbeddings) -> FAISS | None`

**1. Tên hàm + Input**
- `embedding: HuggingFaceEmbeddings` — model embedding đã tạo ở A.1 (phải
  cùng model đã dùng để xây index trước đó, nếu không vector sẽ không tương
  thích không gian).

**2. Output**
- `FAISS | None` — đối tượng vector store của `langchain_community` nếu
  `data/vectorstore/index.faiss` **và** `index.pkl` đều tồn tại và load
  thành công; `None` nếu thiếu file hoặc load lỗi (lỗi được log, không ném
  exception).

**3. Chức năng**
- Tải index FAISS đã lưu trên đĩa vào bộ nhớ, sẵn sàng cho `add_documents`
  (Phần A.10) hoặc truy vấn (Phần C).

**4. Tech stack**
- **`langchain_community.vectorstores.FAISS.load_local(..., allow_dangerous_deserialization=True)`**:
  deserialize index bằng `pickle` — cờ "dangerous" chỉ an toàn vì
  `data/vectorstore/` luôn do chính pipeline này ghi ra, không nhận từ nguồn
  ngoài.

In [5]:
from src.retrieval.vector_store import load_vectorstore

vs = load_vectorstore(embedding)
print("Loại:", type(vs).__name__ if vs is not None else None)
print("Số vector hiện có (trước khi thêm MSFT):", vs.index.ntotal if vs is not None else 0)

2026-08-21 19:29:00 | INFO     | src.retrieval.vector_store | Loaded FAISS index from C:\Python\Personal Project\Financial Report Assistant\data\vectorstore


Loại: FAISS
Số vector hiện có (trước khi thêm MSFT): 720


## A.5 — Kiểm tra idempotent: `read_manifest()` + `indexed_doc_ids(vs)`

Hai hàm này cùng phục vụ **một mục đích**: xác định filing đã được index hay
chưa, để `index_filing()` **bỏ qua (skip)** nếu có — tránh tải lại/embed lại.

**`read_manifest() -> dict[str, Any]`**
- Input: không có.
- Output: toàn bộ nội dung `data/manifest.json`, dạng `{doc_id: entry}`
  (dict rỗng nếu file chưa tồn tại).
- Tech stack: `json` thuần (đọc file).

**`indexed_doc_ids(vs: FAISS | None) -> set[str]`**
- Input: `vs` — kết quả của A.4 (có thể `None`).
- Output: `set[str]` các `doc_id` phân biệt đang thực sự có mặt trong FAISS
  (quét `vs.docstore._dict.values()`, lấy `metadata["doc_id"]` từng chunk).
  Trả `set()` nếu `vs is None`.
- Tech stack: duyệt thuần Python trên cấu trúc nội bộ `InMemoryDocstore` của
  FAISS wrapper.

**Vì sao cần cả hai, không chỉ một?** Điều kiện skip là **OR**:
`doc_id in read_manifest() or doc_id in indexed_doc_ids(vs)`. `manifest.json`
là nơi phục hồi metadata không suy ra được (Phần B); còn chính FAISS mới là
nguồn xác định filing nào **thực sự truy vấn được**.

In [6]:
from src.utils.helpers import read_manifest
from src.retrieval.vector_store import indexed_doc_ids

manifest_before = read_manifest()
indexed_before = indexed_doc_ids(vs)

print("doc_id của MSFT:", ref_msft.doc_id)
print("Có trong manifest.json?      ", ref_msft.doc_id in manifest_before)
print("Có trong FAISS docstore?     ", ref_msft.doc_id in indexed_before)
print("=> Kết luận: MSFT CHƯA được index, sẽ tiếp tục các bước dưới đây.")

doc_id của MSFT: MSFT_10-K_2026-07-29
Có trong manifest.json?       False
Có trong FAISS docstore?      False
=> Kết luận: MSFT CHƯA được index, sẽ tiếp tục các bước dưới đây.


## A.6 — Dataclass `FilingProfile` + `get_profile(filing_type: str) -> FilingProfile`

**Dataclass `FilingProfile`** (`src/preprocessing/filing_profiles.py`,
`@dataclass(frozen=True)`) — chiến lược nhận diện section theo từng loại
form SEC:

| Field | Kiểu | Ý nghĩa |
|---|---|---|
| `filing_type` | `str` | `"10-K"`, `"10-Q"`, hoặc `"GENERIC"` |
| `item_patterns` | `list[re.Pattern]` | Các regex nhận diện tiêu đề Item của form này (rỗng với `GENERIC`) |
| `uses_parts` | `bool` | Có cần gắn nhãn "Part I/II" vào section hay không (chỉ `True` với 10-Q) |
| `part_pattern` | `re.Pattern \| None` | Regex nhận diện tiêu đề "PART I"/"PART II" |
| `min_content_chars` | `int` | Số ký tự nội dung tối thiểu để một tiêu đề được công nhận là section thật |

**`get_profile(filing_type)`**
- Input: `filing_type: str`.
- Output: `FilingProfile` tương ứng (`TEN_K`, `TEN_Q`), hoặc `GENERIC` (kèm
  log cảnh báo) nếu `filing_type` chưa được đăng ký.
- Chức năng: 10-K và 10-Q đánh số Item khác nhau — 10-Q dùng lại Item 1-4
  dưới **cả hai** Part I và Part II (Part I Item 1 = Financial Statements,
  Part II Item 1 = Legal Proceedings) — nên cần chiến lược khác nhau để
  section string không bị đụng độ.
- Tech stack: thuần Python — `dataclass` + `dict` tra cứu, không phụ thuộc
  thư viện ngoài.

In [7]:
from src.preprocessing.filing_profiles import get_profile

profile_10k = get_profile("10-K")
print("filing_type       :", profile_10k.filing_type)
print("uses_parts         :", profile_10k.uses_parts)
print("min_content_chars  :", profile_10k.min_content_chars)
print("Số item_patterns   :", len(profile_10k.item_patterns))

filing_type       : 10-K
uses_parts         : False
min_content_chars  : 1
Số item_patterns   : 23


## A.7 — `parse_html(raw_html_path: str | Path, filing_type: str | None = None) -> Path`

**1. Tên hàm + Input**
- `raw_html_path: str | Path` — đường dẫn file HTML gốc, `data/raw/{doc_id}.html`
  (chính là `ref_msft.raw_path` từ A.3).
- `filing_type: str | None = None` — nếu để `None` (trường hợp bình thường),
  hàm tự suy ra từ tên file qua `parse_doc_id(path.stem)["filing_type"]`, rồi
  gọi `get_profile()` (A.6) để lấy `FilingProfile`.

**2. Output**
- `Path` — đường dẫn file Markdown đã ghi, `data/processed/{doc_id}.md`.

**3. Chức năng**
- Làm sạch HTML (bỏ `<script>`/`<style>`/`<nav>`, phần tử ẩn, khối Mục lục),
  nhận diện tiêu đề section theo `FilingProfile`, chuyển bảng tài chính sang
  bảng Markdown (GFM), rồi ghi ra Markdown với đúng **hợp đồng H2**: mọi
  tiêu đề section thật là một dòng `## `, không gì khác dùng H2; nội dung
  trước tiêu đề đầu tiên được gom vào `## Preamble`.
- **Không tạo `Document` object, không gắn metadata** — đây là ranh giới
  kiến trúc rõ ràng: `parse_html` chỉ sinh ra một file Markdown thuần, có
  thể đọc lại bất kỳ lúc nào (chính điều này làm `--force-rebuild` — Phần B
  — khả thi).

**4. Tech stack**
- **`BeautifulSoup` + `lxml`**: parse cây DOM của HTML (kể cả HTML kiểu
  inline-XBRL của SEC).
- **`FilingProfile`** (A.6): cung cấp regex + ngưỡng để phân biệt tiêu đề
  section thật với văn bản thường.

In [8]:
from src.preprocessing.parse_html import parse_html

md_path_msft = parse_html(ref_msft.raw_path)
print("Ghi ra:", md_path_msft)

text = md_path_msft.read_text(encoding="utf-8")
h2_lines = [line for line in text.splitlines() if line.startswith("## ")]
print(f"Số section (H2) nhận diện được: {len(h2_lines)}")
print("5 section đầu tiên:")
for line in h2_lines[:5]:
    print(" ", line)

2026-08-21 19:29:00 | INFO     | src.preprocessing.parse_html | Parsing C:\Python\Personal Project\Financial Report Assistant\data\raw\MSFT_10-K_2026-07-29.html (profile=10-K)


2026-08-21 19:29:01 | INFO     | src.preprocessing.parse_html | Structural heading detection not confident; falling back to regex.


2026-08-21 19:29:01 | INFO     | src.preprocessing.parse_html | Wrote C:\Python\Personal Project\Financial Report Assistant\data\processed\MSFT_10-K_2026-07-29.md (23 confirmed sections)


Ghi ra: C:\Python\Personal Project\Financial Report Assistant\data\processed\MSFT_10-K_2026-07-29.md
Số section (H2) nhận diện được: 24
5 section đầu tiên:
  ## Preamble
  ## ITEM 1. B USINESS
  ## ITEM 1A. RIS K FACTORS
  ## ITEM 1B. UNRESOLVE D STAFF COMMENTS
  ## ITEM 1C. CY BERSECURITY


## A.8 — Kiểu dữ liệu `Document` + schema metadata §4.3

Trước khi giới thiệu `chunk_markdown` (hàm đầu tiên **tạo ra** `Document`),
cần hiểu `Document` là gì — vì nó là kiểu dữ liệu trung tâm xuyên suốt mọi
luồng còn lại (chunk, index, retrieve, cite).

`Document` (`langchain_core.documents.Document`) là một mô hình dữ liệu của
LangChain với 2 trường:

| Field | Kiểu | Ý nghĩa |
|---|---|---|
| `page_content` | `str` | Nội dung văn bản thật của chunk (dùng để embed + hiển thị) |
| `metadata` | `dict` | Thông tin đi kèm, dùng để lọc/trích dẫn, **không dùng để embed** |

Trong dự án này, `metadata` của **mọi** chunk luôn có đúng 9 khoá (schema
§4.3, được tạo trong `chunking.py`, không phải `parse_html.py`):

| Khoá | Ý nghĩa |
|---|---|
| `doc_id` | Khoá định danh filing |
| `ticker` | Mã cổ phiếu |
| `filing_type` | Loại form |
| `filing_date` | Ngày nộp |
| `source` | Tên file HTML gốc |
| `filing_name` | Tên hiển thị |
| `section` | Tiêu đề H2 chính xác (đã gắn "Part..." nếu là 10-Q) |
| `url` | URL SEC gốc |
| `chunk_id` | Số thứ tự chunk, liên tục từ 0 xuyên suốt cả filing |

## A.9 — `chunk_markdown(md_path: str | Path, filing_meta: dict) -> list[Document]`

**1. Tên hàm + Input**
- `md_path: str | Path` — file Markdown từ A.7 (`md_path_msft`).
- `filing_meta: dict` — các trường **không suy ra được** cần gắn vào mọi
  chunk: `doc_id`, `ticker`, `filing_type`, `filing_date`, `filing_name`,
  `source`, `url`. Trong luồng CLI, dict này được tạo bằng
  `dataclasses.asdict(ref)` — chuyển thẳng `FilingRef` (A.2) thành `dict`
  (các khoá thừa như `cik`/`accession`/`raw_path` bị bỏ qua, không gây lỗi).

**2. Output**
- `list[Document]` (xem A.8) — mỗi phần tử là một chunk, `metadata` đúng 9
  khoá §4.3, `page_content` bắt đầu bằng một **header ngữ cảnh 3 dòng**
  (`Company: .../Filing: .../Section: ...`) rồi một dòng trống rồi mới đến
  nội dung thật — thêm vào để chunk ngắn (vd "Item 1B. Unresolved Staff
  Comments" nhiều khi chỉ có "None.") vẫn mang đủ ngữ cảnh công ty/filing
  khi embed.

**3. Chức năng**
- Tách Markdown theo đúng `^##\s+` (không bao giờ cắt ngang một section),
  chạy `RecursiveCharacterTextSplitter` **độc lập theo từng section** (nên
  một chunk không bao giờ trộn 2 section khác nhau), rồi gắn header ngữ
  cảnh **sau khi** đã cắt (để header không tính vào `chunk_size`).

**4. Tech stack**
- **`langchain_text_splitters.RecursiveCharacterTextSplitter`**
  (`chunk_size=1000, chunk_overlap=200`): cắt văn bản dài theo ranh giới
  đoạn/câu/từ, có chồng lấn để không mất ngữ cảnh ở biên chunk.
- **`langchain_core.documents.Document`**: kiểu output chuẩn hoá, để mọi
  bước sau (embed, FAISS, retriever) xử lý đồng nhất bất kể nguồn.

In [9]:
from src.preprocessing.chunking import chunk_markdown

filing_meta_msft = asdict(ref_msft)
docs_msft = chunk_markdown(md_path_msft, filing_meta=filing_meta_msft)

print(f"Số chunk sinh ra: {len(docs_msft)}")
print()
print("--- Document đầu tiên ---")
print("metadata:")
pprint(docs_msft[0].metadata)
print()
print("page_content (300 ký tự đầu):")
print(docs_msft[0].page_content[:300])

2026-08-21 19:29:01 | INFO     | src.preprocessing.chunking | Chunking C:\Python\Personal Project\Financial Report Assistant\data\processed\MSFT_10-K_2026-07-29.md


2026-08-21 19:29:01 | INFO     | src.preprocessing.chunking | Produced 520 chunks across 24 sections for doc_id=MSFT_10-K_2026-07-29


Số chunk sinh ra: 520

--- Document đầu tiên ---
metadata:
{'chunk_id': 0,
 'doc_id': 'MSFT_10-K_2026-07-29',
 'filing_date': '2026-07-29',
 'filing_name': 'MICROSOFT CORP 10-K 2026',
 'filing_type': '10-K',
 'section': 'Preamble',
 'source': 'MSFT_10-K_2026-07-29.html',
 'ticker': 'MSFT',
 'url': 'https://www.sec.gov/Archives/edgar/data/789019/000119312526323660/msft-20260630.htm'}

page_content (300 ký tự đầu):
Company: MICROSOFT CORP 10-K 2026 (MSFT)
Filing: 10-K filed 2026-07-29
Section: Preamble

UNITED STATES
SECURITIES AND EXCHANGE COMMISSION
Washington, D.C. 20549
FORM 10-K
|  |  |
| --- | --- |
| ☒ | ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934 |
|  |  |
| ---


## A.10 — `update_vectorstore(vs: FAISS | None, documents: list[Document], embedding) -> FAISS`

**1. Tên hàm + Input**
- `vs: FAISS | None` — vector store hiện có, từ A.4 (đã có 720 vector của
  NVDA 10-K + 10-Q).
- `documents: list[Document]` — danh sách chunk mới, từ A.9.
- `embedding` — model embedding từ A.1 (**điểm inject embedding thứ 2**
  trong luồng — lần này để tính vector cho chunk mới).

**2. Output**
- `FAISS` — vector store đã cập nhật (nếu `vs` khác `None`, đây chính là
  **cùng object** `vs` đã được sửa tại chỗ qua `add_documents`; nếu `vs`
  là `None`, một index hoàn toàn mới được dựng).

**3. Chức năng**
- Nếu đã có index: gọi `vs.add_documents(documents)` — LangChain tự động
  gọi `embedding.embed_documents(...)` trên từng `page_content` rồi thêm
  vector vào FAISS. Nếu chưa có: dựng mới hoàn toàn (`build_vectorstore`).
- **Không tự lưu ra đĩa** — caller phải tự gọi `save_vectorstore` (A.11).

**4. Tech stack**
- **`langchain_community.vectorstores.FAISS.add_documents`**: bọc quanh
  `faiss` (Facebook AI Similarity Search) — thêm vector vào index dạng
  `IndexFlat` (tìm kiếm brute-force chính xác, không xấp xỉ).

In [10]:
from src.retrieval.vector_store import update_vectorstore

n_before = vs.index.ntotal
vs = update_vectorstore(vs, docs_msft, embedding)
n_after = vs.index.ntotal

print(f"Số vector trước: {n_before}")
print(f"Số vector sau:   {n_after}")
print(f"Chênh lệch:      {n_after - n_before} (= số chunk MSFT vừa thêm: {len(docs_msft)})")

2026-08-21 19:29:39 | INFO     | src.retrieval.vector_store | Added 520 documents to existing FAISS index


Số vector trước: 720
Số vector sau:   1240
Chênh lệch:      520 (= số chunk MSFT vừa thêm: 520)


## A.11 — `save_vectorstore(vs: FAISS) -> None`

**1. Tên hàm + Input**
- `vs: FAISS` — vector store đã cập nhật, từ A.10.

**2. Output**
- `None` — hàm chỉ có tác dụng phụ (ghi file), không trả giá trị.

**3. Chức năng**
- Ghi đè `data/vectorstore/index.faiss` và `data/vectorstore/index.pkl`
  bằng nội dung hiện tại của `vs` (đã có cả NVDA lẫn MSFT).
- `.faiss` chứa vector kèm index, `.pkl` chứa nội dung và metadata, ta có thể xác định liên kết giữa nội dung và vector thông qua hàm `index_to_docstore_id`

**4. Tech stack**
- **`FAISS.save_local(...)`**: serialize hoá index (phần vector nhị phân
  qua chính thư viện `faiss`, phần docstore + metadata qua `pickle`).

In [11]:
from src.retrieval.vector_store import save_vectorstore
from src.utils.config import VECTORSTORE_DIR

save_vectorstore(vs)
print("Đã ghi:")
for name in ("index.faiss", "index.pkl"):
    p = VECTORSTORE_DIR / name
    print(f"  {p}  (tồn tại: {p.exists()}, {p.stat().st_size:,} bytes)")

2026-08-21 19:29:39 | INFO     | src.retrieval.vector_store | Saved FAISS index to C:\Python\Personal Project\Financial Report Assistant\data\vectorstore


Đã ghi:
  C:\Python\Personal Project\Financial Report Assistant\data\vectorstore\index.faiss  (tồn tại: True, 1,904,685 bytes)
  C:\Python\Personal Project\Financial Report Assistant\data\vectorstore\index.pkl  (tồn tại: True, 1,342,767 bytes)


## A.12 — `upsert_manifest(doc_id: str, entry: dict) -> None`

**1. Tên hàm + Input**
- `doc_id: str` — khoá, ở đây là `ref_msft.doc_id`.
- `entry: dict` — các trường không suy ra được cần lưu bền: `ticker`,
  `filing_type`, `filing_date`, `filing_name`, `source`, `url`, `cik`,
  `accession`, `indexed_at` (thời điểm index, ISO 8601 UTC).

**2. Output**
- `None` — tác dụng phụ ghi file.

**3. Chức năng**
- Đọc `data/manifest.json` hiện có, gộp thêm/ghi đè entry của `doc_id` này,
  ghi lại toàn bộ file. Đây là bước **cuối cùng** của luồng index — và chỉ
  được gọi (trong `index_filing`, xem A.14) nếu A.11 đã thành công, để
  `manifest.json` không bao giờ "nói dối" là đã index xong trong khi FAISS
  chưa thực sự lưu.

**4. Tech stack**
- **`json`** thuần: đọc-sửa-ghi (read-modify-write) file `data/manifest.json`.

In [12]:
from src.utils.helpers import upsert_manifest

entry_msft = {
    "ticker": ref_msft.ticker,
    "filing_type": ref_msft.filing_type,
    "filing_date": ref_msft.filing_date,
    "filing_name": ref_msft.filing_name,
    "source": ref_msft.source,
    "url": ref_msft.url,
    "cik": ref_msft.cik,
    "accession": ref_msft.accession,
    "indexed_at": datetime.now(timezone.utc).isoformat(),
}
upsert_manifest(ref_msft.doc_id, entry_msft)

manifest_now = read_manifest()
print("Các doc_id hiện có trong manifest.json:")
for doc_id in manifest_now:
    print(" -", doc_id)

2026-08-21 19:29:40 | INFO     | src.utils.helpers | Manifest updated for doc_id=MSFT_10-K_2026-07-29


Các doc_id hiện có trong manifest.json:
 - NVDA_10-K_2026-02-25
 - NVDA_10-Q_2026-05-20
 - MSFT_10-K_2026-07-29


### Tổng kết: đã index xong MSFT — hoàn toàn thủ công, từng hàm một

Từ A.1 đến A.12, ta vừa tái hiện chính xác những gì `index_filing()` sẽ làm
bên trong — gọi 9 hàm theo đúng thứ tự phụ thuộc, mỗi hàm nhận input là
output của hàm trước. Bây giờ mới giới thiệu `index_filing()` — hàm điều
phối duy nhất gói toàn bộ lại.

## A.13 — `index_filing(ticker: str, filing_type: str, force_rebuild: bool = False) -> dict`

**1. Tên hàm + Input**
- `ticker: str` — mã cổ phiếu (bỏ qua nếu `force_rebuild=True`).
- `filing_type: str` — form SEC (bỏ qua nếu `force_rebuild=True`).
- `force_rebuild: bool = False` — nếu `True`, rẽ sang **luồng B** (Phần B)
  thay vì luồng này.

**2. Output**
- `dict` dạng `{"doc_id": str | None, "status": "indexed" | "skipped" | "rebuilt", "chunk_count": int}`.
- Ném exception (`ValueError`/`RuntimeError`) nếu thất bại — không trả một
  "status lỗi" thầm lặng.

**3. Chức năng**
- **Đây là luồng code index duy nhất trong toàn dự án** — gọi đúng các hàm
  A.1, A.3–A.12 theo thứ tự, cộng thêm 2 cơ chế bảo vệ:
  - **Guard chặn rỗng**: nếu `chunk_markdown` (A.9) trả về `[]`, ném
    `ValueError` ngay, **không bao giờ** gọi `update_vectorstore` với danh
    sách rỗng.
  - **Dọn "sticky download"**: nếu HTML vừa được **tải mới** trong chính lần
    gọi này (không phải tái sử dụng cache) mà bước parse/chunk sau đó thất
    bại, file `data/raw/{doc_id}.html` bị xoá — để lần thử lại tải file mới
    thay vì dùng mãi một file lỗi/rỗng đã tải hỏng. Nếu HTML được tái sử
    dụng từ lần chạy trước, file được giữ nguyên bất kể lỗi.
- Được gọi bởi **cả 2 nơi**: `main.py:main()` (CLI, luồng A/B) **và**
  `app/streamlit_app.py:_render_indexing_form()` (Phần D) — không có bản
  sao logic thứ hai.

**4. Tech stack**
- Thuần điều phối (orchestration) — không tự dùng thư viện ngoài nào ngoài
  các hàm đã giới thiệu ở A.1–A.12.

### Minh hoạ 1 — gọi lại cho MSFT (đã index thủ công ở trên) → phải **skip**

In [13]:
from main import index_filing

result_msft_again = index_filing("MSFT", "10-K")
pprint(result_msft_again)
assert result_msft_again["status"] == "skipped", "Đáng lẽ phải skip vì MSFT vừa được index thủ công ở trên!"
print()
print("=> Đúng như kỳ vọng: index_filing() nhận ra MSFT đã có trong manifest + FAISS, không tải/embed lại.")

2026-08-21 19:29:40 | INFO     | src.embeddings.embedder | Loading embedding model: BAAI/bge-small-en-v1.5


2026-08-21 19:29:43 | INFO     | src.utils.helpers | GET https://data.sec.gov/submissions/CIK0000789019.json


2026-08-21 19:29:44 | INFO     | src.ingestion.sec_loader | Raw filing already cached for doc_id=MSFT_10-K_2026-07-29 (C:\Python\Personal Project\Financial Report Assistant\data\raw\MSFT_10-K_2026-07-29.html); skipping download.


2026-08-21 19:29:44 | INFO     | src.retrieval.vector_store | Loaded FAISS index from C:\Python\Personal Project\Financial Report Assistant\data\vectorstore


2026-08-21 19:29:44 | INFO     | main | doc_id=MSFT_10-K_2026-07-29 is already indexed; skipping.


{'chunk_count': 0, 'doc_id': 'MSFT_10-K_2026-07-29', 'status': 'skipped'}

=> Đúng như kỳ vọng: index_filing() nhận ra MSFT đã có trong manifest + FAISS, không tải/embed lại.


### Minh hoạ 2 — index **GOOGL** hoàn toàn mới, chỉ bằng 1 lệnh duy nhất

Đối lập với cách làm thủ công ở A.1–A.12, lần này gọi `index_filing()` cho
một ticker **chưa từng index** — toàn bộ chuỗi tải → parse → chunk → embed →
lưu → ghi manifest diễn ra bên trong một lệnh gọi.

In [14]:
t0 = time.time()
result_googl = index_filing("GOOGL", "10-K")
elapsed = time.time() - t0

pprint(result_googl)
print(f"\nThời gian: {elapsed:.1f}s")
assert result_googl["status"] == "indexed"

2026-08-21 19:29:44 | INFO     | src.embeddings.embedder | Loading embedding model: BAAI/bge-small-en-v1.5


2026-08-21 19:29:47 | INFO     | src.utils.helpers | GET https://data.sec.gov/submissions/CIK0001652044.json


2026-08-21 19:29:47 | INFO     | src.ingestion.sec_loader | Downloading filing doc_id=GOOGL_10-K_2026-02-05 from https://www.sec.gov/Archives/edgar/data/1652044/000165204426000018/goog-20251231.htm


2026-08-21 19:29:47 | INFO     | src.utils.helpers | GET https://www.sec.gov/Archives/edgar/data/1652044/000165204426000018/goog-20251231.htm


2026-08-21 19:29:48 | INFO     | src.retrieval.vector_store | Loaded FAISS index from C:\Python\Personal Project\Financial Report Assistant\data\vectorstore


...

2026-08-21 19:30:23 | INFO     | src.retrieval.vector_store | Added 514 documents to existing FAISS index


2026-08-21 19:30:23 | INFO     | src.retrieval.vector_store | Saved FAISS index to C:\Python\Personal Project\Financial Report Assistant\data\vectorstore


2026-08-21 19:30:23 | INFO     | src.utils.helpers | Manifest updated for doc_id=GOOGL_10-K_2026-02-05


2026-08-21 19:30:23 | INFO     | main | Indexed doc_id=GOOGL_10-K_2026-02-05 (514 chunks).


{'chunk_count': 514, 'doc_id': 'GOOGL_10-K_2026-02-05', 'status': 'indexed'}

Thời gian: 39.7s


---
# PHẦN B — Luồng Force-rebuild

**Kích hoạt bởi:** `python main.py --ticker <bất_kỳ> --filing <bất_kỳ> --force-rebuild`
(tham số `--ticker`/`--filing` bị **bỏ qua hoàn toàn** trong luồng này, dù
CLI vẫn bắt buộc phải truyền).

## Các bên tham gia — khác biệt lớn nhất so với Phần A

```
User (terminal)
   │  gõ lệnh CLI --force-rebuild
   ▼
main.py (CLI)
   │
   ▼
File System (đọc TẤT CẢ data/processed/*.md + data/manifest.json)
   │           ← KHÔNG gọi SEC EDGAR, hoàn toàn offline
   ▼
FAISS Vector Store (dựng MỚI HOÀN TOÀN, ghi đè data/vectorstore/*)
```

**Vì sao cần luồng này riêng?** FAISS (`IndexFlat`) không hỗ trợ xoá/sửa
từng vector một cách ổn định — nên cách duy nhất để "sửa" index (vd sau khi
đổi cách chunk, đổi `chunk_size`, hay đơn giản là nghi ngờ index bị lệch với
Markdown) là dựng lại **toàn bộ từ đầu**, không sửa từng phần.

## Vì sao Markdown một mình không đủ — phải phục hồi từ manifest

`data/processed/{doc_id}.md` (Phần A.7) **chỉ chứa** tiêu đề H1 và nội dung
theo section H2 — theo đúng nguyên tắc "`parse_html.py` không gắn metadata".
File này **không hề chứa** `ticker`/`filing_type`/`filing_date`/`filing_name`/`url`
ở bất kỳ đâu. Vì `chunk_markdown` (A.9) bắt buộc đủ 7 khoá này, và chỉ
`doc_id` suy ra được từ tên file, nên `url` và `filing_name` **chỉ có thể**
phục hồi từ `data/manifest.json` — không phải bằng cách re-parse Markdown,
càng không phải gọi lại SEC API (điều mà `--force-rebuild` cố tình tránh, để
là một thao tác nhanh và offline).

In [15]:
from src.utils.config import PROCESSED_DIR

md_files = sorted(PROCESSED_DIR.glob("*.md"))
print(f"Hiện có {len(md_files)} file Markdown trong data/processed/:")
for p in md_files:
    print(" -", p.name)

Hiện có 4 file Markdown trong data/processed/:
 - GOOGL_10-K_2026-02-05.md
 - MSFT_10-K_2026-07-29.md
 - NVDA_10-K_2026-02-25.md
 - NVDA_10-Q_2026-05-20.md


## B.1 — `manifest_get(doc_id: str) -> dict[str, Any] | None`

Khác với `read_manifest()` (A.5, lấy **toàn bộ** manifest), hàm này lấy
**đúng một entry**.

**1. Tên hàm + Input**
- `doc_id: str` — khoá cần tra, vd `"NVDA_10-K_2026-02-25"`.

**2. Output**
- `dict[str, Any] | None` — entry của riêng `doc_id` đó (các trường
  `ticker`/`filing_type`/`filing_date`/`filing_name`/`source`/`url`/`cik`/
  `accession`/`indexed_at`), hoặc `None` nếu `doc_id` không có trong manifest.

**3. Chức năng**
- Với **mỗi** file `data/processed/*.md`, `_force_rebuild` (bên trong
  `index_filing`) gọi hàm này để dựng lại `filing_meta` = `{"doc_id": doc_id, **entry}`,
  rồi gọi lại **chính hàm `chunk_markdown`** đã giới thiệu ở A.9 — không có
  hàm chunk thứ hai nào khác cho luồng rebuild.
- Nếu một file `.md` không có entry tương ứng trong manifest (vd bị xoá thủ
  công), file đó bị **bỏ qua** (log cảnh báo), không làm hỏng cả quá trình
  rebuild.

**4. Tech stack**
- `json` thuần — gọi `read_manifest().get(doc_id)`.

In [16]:
from src.utils.helpers import manifest_get

for md_file in md_files:
    doc_id = md_file.stem
    entry = manifest_get(doc_id)
    status = "✓ có trong manifest" if entry else "✗ THIẾU trong manifest (sẽ bị bỏ qua khi rebuild)"
    print(f"{doc_id:30s} {status}")

GOOGL_10-K_2026-02-05          ✓ có trong manifest
MSFT_10-K_2026-07-29           ✓ có trong manifest
NVDA_10-K_2026-02-25           ✓ có trong manifest
NVDA_10-Q_2026-05-20           ✓ có trong manifest


## B.2 — `index_filing(force_rebuild=True)` — tái sử dụng hàm đã giới thiệu ở A.13

Không có hàm mới nào ở bước này — chỉ là **nhánh khác** của cùng hàm
`index_filing()` đã giới thiệu ở A.13. Bên trong, nhánh này:

1. Liệt kê toàn bộ `data/processed/*.md`.
2. Với mỗi file, phục hồi `filing_meta` qua `manifest_get` (B.1) rồi gọi
   `chunk_markdown` (A.9).
3. Gộp **tất cả** chunk của **tất cả** file lại thành một danh sách duy nhất.
4. Gọi `build_vectorstore(all_docs, embedding)` — **dựng FAISS mới hoàn
   toàn** (khác với A.10 `update_vectorstore`, vốn tải index cũ rồi thêm
   vào) — rồi `save_vectorstore` (A.11), **ghi đè** `data/vectorstore/*`.
5. **Không gọi `upsert_manifest`** — manifest chỉ được **đọc**, không được
   ghi trong nhánh này (vì không có filing mới nào được "index" theo nghĩa
   phát hiện lần đầu, chỉ là dựng lại từ dữ liệu đã có).

In [17]:
n_before_rebuild = vs.index.ntotal

t0 = time.time()
result_rebuild = index_filing("", "", force_rebuild=True)
elapsed = time.time() - t0

pprint(result_rebuild)
print(f"\nThời gian rebuild: {elapsed:.1f}s")
assert result_rebuild["status"] == "rebuilt"
assert result_rebuild["doc_id"] is None, "force-rebuild liên quan nhiều filing, không phải một doc_id"

2026-08-21 19:30:23 | INFO     | src.embeddings.embedder | Loading embedding model: BAAI/bge-small-en-v1.5


2026-08-21 19:30:27 | INFO     | src.preprocessing.chunking | Chunking C:\Python\Personal Project\Financial Report Assistant\data\processed\GOOGL_10-K_2026-02-05.md


2026-08-21 19:30:27 | INFO     | src.preprocessing.chunking | Produced 514 chunks across 24 sections for doc_id=GOOGL_10-K_2026-02-05


2026-08-21 19:30:27 | INFO     | main | Recovered 514 chunks for doc_id=GOOGL_10-K_2026-02-05 from manifest.


2026-08-21 19:30:27 | INFO     | src.preprocessing.chunking | Chunking C:\Python\Personal Project\Financial Report Assistant\data\processed\MSFT_10-K_2026-07-29.md


...

2026-08-21 19:30:27 | INFO     | main | Recovered 205 chunks for doc_id=NVDA_10-Q_2026-05-20 from manifest.


2026-08-21 19:30:27 | INFO     | src.retrieval.vector_store | Building new FAISS index from 1754 documents


2026-08-21 19:32:22 | INFO     | src.retrieval.vector_store | Saved FAISS index to C:\Python\Personal Project\Financial Report Assistant\data\vectorstore


2026-08-21 19:32:22 | INFO     | main | Rebuilt FAISS index from 4 file(s), 1754 total chunks.


{'chunk_count': 1754, 'doc_id': None, 'status': 'rebuilt'}

Thời gian rebuild: 118.4s


In [18]:
# Xác nhận: url/filing_name của TỪNG filing đã sống sót qua vòng round-trip qua manifest
vs_rebuilt = load_vectorstore(embedding)
print(f"Số vector sau rebuild: {vs_rebuilt.index.ntotal} (trước rebuild trong bộ nhớ: {n_before_rebuild})")
print()

seen_doc_ids = {}
for doc in vs_rebuilt.docstore._dict.values():
    doc_id = doc.metadata["doc_id"]
    if doc_id not in seen_doc_ids:
        seen_doc_ids[doc_id] = doc.metadata

for doc_id, meta in seen_doc_ids.items():
    ok = bool(meta.get("url")) and bool(meta.get("filing_name"))
    print(f"{doc_id:28s} url/filing_name còn nguyên: {ok}   ({meta['filing_name']})")

2026-08-21 19:32:22 | INFO     | src.retrieval.vector_store | Loaded FAISS index from C:\Python\Personal Project\Financial Report Assistant\data\vectorstore


Số vector sau rebuild: 1754 (trước rebuild trong bộ nhớ: 1240)

GOOGL_10-K_2026-02-05        url/filing_name còn nguyên: True   (Alphabet Inc. 10-K 2026)
MSFT_10-K_2026-07-29         url/filing_name còn nguyên: True   (MICROSOFT CORP 10-K 2026)
NVDA_10-K_2026-02-25         url/filing_name còn nguyên: True   (NVIDIA CORP 10-K 2026)
NVDA_10-Q_2026-05-20         url/filing_name còn nguyên: True   (NVIDIA CORP 10-Q 2026)


---
# PHẦN C — Luồng Truy vấn (Query)

**Kích hoạt bởi:** người dùng gõ câu hỏi trên Streamlit chat (trong notebook
này ta gọi thẳng các hàm nền — không cần chạy Streamlit server, vì luồng
này **không** phụ thuộc bất kỳ thứ gì đặc thù của Streamlit).

## Các bên tham gia

```
User
   │  câu hỏi + bộ lọc (filters)
   ▼
Retriever  ──────────────►  FAISS Vector Store (MMR search)
   │  ◄───────────────────────────┘  list[Document]
   ▼
RAG Chain (LCEL)  ──────────────►  Gemini API (Google)
   │  ◄─────────────────────────────────┘  câu trả lời text
   ▼
User (câu trả lời + trích dẫn (citations))
```

## C.1 — `create_retriever(vectorstore: VectorStore, k: int = 5, fetch_k: int = 20) -> Retriever`

**1. Tên hàm + Input**
- `vectorstore: VectorStore` — FAISS đã có sẵn (từ Phần A/B, biến `vs_rebuilt`).
- `k: int = 5` (`RETRIEVAL_K`) — số document tối đa trả về mỗi lần truy vấn.
- `fetch_k: int = 20` (`FETCH_K`) — kích thước pool ứng viên MMR xem xét
  trước khi chọn ra `k`.

**2. Output**
- `Retriever` — một object (không phải dataclass, là một class thường) với
  3 thuộc tính chính: `_vectorstore`, `_k`, `_fetch_k`, `_pool_multiplier`,
  và `_base_retriever` (một `VectorStoreRetriever` của LangChain dựng sẵn
  với `search_type="mmr"`). Có đúng một phương thức public:
  `.retrieve(question, filters=None) -> list[Document]`.

**3. Chức năng**
- Đóng gói toàn bộ logic MMR + lọc metadata (xem C.2) đằng sau một API đơn
  giản, để `rag_chain.py` (C.4) và `metrics.py` (Phần E) dùng chung mà
  không cần biết chi tiết FAISS bên dưới.

**4. Tech stack**
- **`vectorstore.as_retriever(search_type="mmr", ...)`**: dùng thuật toán
  **Maximal Marginal Relevance** của LangChain — chọn `k` kết quả vừa liên
  quan vừa **đa dạng** (tránh 5 chunk gần giống hệt nhau), thay vì chỉ lấy
  top-k theo độ tương đồng thuần tuý.

In [19]:
from src.retrieval.retriever import create_retriever

retriever = create_retriever(vs_rebuilt)
print("k       :", retriever._k)
print("fetch_k :", retriever._fetch_k)

k       : 5
fetch_k : 20


## C.2 — `Retriever.retrieve(question: str, filters: dict | None = None) -> list[Document]`

**1. Tên hàm + Input**
- `question: str` — câu hỏi người dùng, dạng văn bản tự nhiên.
- `filters: dict | None = None` — điều kiện lọc metadata chính xác tuyệt
  đối theo từng khoá, vd `{"ticker": "NVDA"}` hoặc
  `{"ticker": "NVDA", "filing_type": "10-Q"}`.

**2. Output**
- `list[Document]` — tối đa `k` document, **KHÔNG lọc = không loại
  trùng khoá section**, sắp theo độ liên quan giảm dần.

**3. Chức năng — hai nhánh rẽ rõ rệt**
- **Không có `filters`** (`None` hoặc `{}`): gọi thẳng
  `self._base_retriever.invoke(question)` — MMR top-k thuần trên toàn bộ
  index, không lọc.
- **Có `filters`**: mở rộng pool ứng viên lên
  `fetch_k * FILTER_POOL_MULTIPLIER` (mặc định `20 × 5 = 100`) bằng
  `vectorstore.max_marginal_relevance_search(...)`, lọc trong bộ nhớ (mọi
  cặp khoá/giá trị trong `filters` phải khớp `Document.metadata` chính
  xác), rồi cắt còn `k` kết quả đầu. Nếu lọc xong còn ít hơn `k`, trả về
  đúng số lượng còn lại — **không "độn" thêm** cho đủ `k`.
- **Vì sao lọc ở đây, không lọc ngay trong FAISS?** `IndexFlat` của FAISS
  không hỗ trợ lọc theo metadata native — nên phải "mở pool rộng ra trước,
  rồi lọc, rồi mới cắt còn k" (pool → filter → top-k), để không bị hụt kết
  quả nếu lọc quá sớm trên một top-k quá nhỏ.

**4. Tech stack**
- **`FAISS.max_marginal_relevance_search`**: cùng thuật toán MMR như C.1
  nhưng cho phép chỉ định `k`/`fetch_k` tuỳ biến ngay tại lúc gọi (dùng để
  "mở pool rộng ra" khi cần lọc).

### Minh hoạ trực tiếp: so sánh nhánh không lọc vs. có lọc

Bây giờ index đã có **3 công ty** (NVDA, MSFT, GOOGL) — phép so sánh này có
ý nghĩa rõ ràng hơn nhiều so với khi chỉ có 1 công ty.

In [20]:
question_demo = "What are the main risk factors disclosed in the filing?"

# Nhánh 1: KHÔNG lọc — có thể trộn lẫn nhiều công ty
docs_no_filter = retriever.retrieve(question_demo)
print(f"--- Không filter: {len(docs_no_filter)} kết quả ---")
for d in docs_no_filter:
    print(f"  ticker={d.metadata['ticker']:6s} section={d.metadata['section']}")

print()

# Nhánh 2: CÓ lọc theo ticker=NVDA — chỉ NVDA được trả về
docs_filtered = retriever.retrieve(question_demo, filters={"ticker": "NVDA"})
print(f"--- Filter ticker=NVDA: {len(docs_filtered)} kết quả ---")
for d in docs_filtered:
    print(f"  ticker={d.metadata['ticker']:6s} section={d.metadata['section']}")
assert all(d.metadata["ticker"] == "NVDA" for d in docs_filtered)

--- Không filter: 5 kết quả ---
  ticker=GOOGL  section=ITEM 1A. RISK FACTORS
  ticker=NVDA   section=Item 1A. Risk Factors
  ticker=GOOGL  section=ITEM 1A. RISK FACTORS
  ticker=GOOGL  section=ITEM 1A. RISK FACTORS
  ticker=GOOGL  section=ITEM 1A. RISK FACTORS



--- Filter ticker=NVDA: 5 kết quả ---
  ticker=NVDA   section=Item 1A. Risk Factors
  ticker=NVDA   section=Item 1A. Risk Factors
  ticker=NVDA   section=Item 1A. Risk Factors
  ticker=NVDA   section=Item 1A. Risk Factors
  ticker=NVDA   section=Item 1A. Risk Factors


## C.3 — `get_llm() -> ChatGoogleGenerativeAI`

**1. Tên hàm + Input**
- Không có tham số — đọc trực tiếp `GEMINI_MODEL`, `GEMINI_API_KEY`,
  `GEMINI_TEMPERATURE` (=0.2) từ `src.utils.config`.

**2. Output**
- `ChatGoogleGenerativeAI` — đối tượng chat model của
  `langchain_google_genai`, sẵn sàng dùng trong một LCEL chain.

**3. Chức năng**
- Khởi tạo model Gemini dùng để sinh câu trả lời — **luôn được inject** vào
  `create_rag_chain` (C.4), không hàm nào khác tự tạo `ChatGoogleGenerativeAI`
  riêng.

**4. Tech stack**
- **`langchain_google_genai.ChatGoogleGenerativeAI`**: wrapper LangChain gọi
  Gemini API của Google qua REST, expose theo interface `BaseChatModel`
  chuẩn của LangChain (để ghép được vào LCEL pipeline chung với mọi chat
  model khác).

In [21]:
from src.generation.rag_chain import get_llm

llm = get_llm()
print("Loại   :", type(llm).__name__)
print("Model  :", llm.model)
print("Temp   :", llm.temperature)

2026-08-21 19:32:22 | INFO     | src.generation.rag_chain | Loading Gemini model: gemini-flash-latest


Loại   : ChatGoogleGenerativeAI
Model  : models/gemini-flash-latest
Temp   : 0.2


## C.4 — `create_rag_chain(retriever: Retriever, llm: BaseChatModel) -> Runnable`

**1. Tên hàm + Input**
- `retriever: Retriever` — từ C.1.
- `llm: BaseChatModel` — từ C.3.

**2. Output**
- `Runnable` — một **đồ thị LCEL** (LangChain Expression Language) nhận
  input `{"question": str, "filters": dict | None}`, trả
  `{"answer": str, "documents": list[Document]}`.

**3. Chức năng — điểm thiết kế quan trọng nhất của cả dự án**
- Đồ thị gồm 3 giai đoạn nối bằng `|`:
  1. `RunnablePassthrough.assign(documents=...)` — gọi
     `retriever.retrieve(question, filters)` **đúng một lần**, thêm kết quả
     vào dict.
  2. `RunnablePassthrough.assign(context=...)` — định dạng `documents` vừa
     lấy thành chuỗi có tiêu đề `--- SOURCE 1 ---`, `--- SOURCE 2 ---`, ...
     (đọc lại **cùng** danh sách `documents`, không truy vấn lại).
  3. `RunnableParallel(answer=..., documents=...)` — chạy song song 2
     nhánh trên **cùng một dict**: nhánh `answer` đưa `context`+`question`
     qua prompt rồi qua `llm`; nhánh `documents` chỉ trả nguyên danh sách đã
     có từ giai đoạn 1.
- Nhờ vậy, **retrieval chỉ chạy đúng 1 lần**, và câu trả lời + trích dẫn
  (Phần C.5) luôn dựa trên **cùng một tập chunk** — không có nguy cơ trả
  lời dựa trên chunk A nhưng trích dẫn chunk B.

**4. Tech stack**
- **LangChain LCEL** (`RunnablePassthrough`, `RunnableParallel`,
  `RunnableLambda`, `ChatPromptTemplate`, `StrOutputParser`): mô hình
  "đường ống" khai báo, cho phép ghép nhiều bước xử lý (retrieve → format →
  gọi LLM → parse) thành một object `Runnable` duy nhất, gọi bằng
  `.invoke(...)`.

In [22]:
from src.generation.rag_chain import create_rag_chain

chain = create_rag_chain(retriever, llm)
print("Loại chain:", type(chain).__name__)

Loại chain: RunnableSequence


## C.5 — `ask(chain: Runnable, question: str, filters: dict | None = None) -> dict`

**1. Tên hàm + Input**
- `chain: Runnable` — từ C.4.
- `question: str` — câu hỏi.
- `filters: dict | None = None` — cùng ý nghĩa như C.2, truyền thẳng xuống
  `retriever.retrieve` bên trong chain.

**2. Output**
- `dict` dạng `{"answer": str, "citations": list[dict], "documents": list[Document]}`.
  - `citations`: các bộ ba `(source, section, chunk_id)` **đã loại trùng**,
    dựng **theo chương trình** trực tiếp từ `metadata` của `documents` —
    **không bao giờ** yêu cầu LLM tự sinh ra trích dẫn (vì model có thể bịa
    hoặc chép sai số — đã quan sát thực tế: model tự nói "chunk 384" trong
    văn bản trả lời trong khi chunk thật được trích là 284).
  - `documents`: chính xác danh sách `Document` đã dùng ở C.4 giai đoạn 1 —
    **không loại trùng**, giữ nguyên thứ tự.

**3. Chức năng**
- Gọi `chain.invoke({"question": question, "filters": filters})`, rồi dựng
  `citations` bằng cách duyệt `documents`, gộp trùng bộ ba
  `(source, section, chunk_id)` qua một `set` (giữ thứ tự xuất hiện đầu
  tiên).

**4. Tech stack**
- Thuần Python (dedup bằng `set`) trên kết quả của LCEL chain (C.4).

### Minh hoạ — 3 câu hỏi thật, qua Gemini thật

In [23]:
from src.generation.rag_chain import ask

result_unfiltered = ask(chain, "What drove NVIDIA revenue growth?")
print("=== Không filter ===")
print(result_unfiltered["answer"])
print()
print("Citations:")
for c in result_unfiltered["citations"]:
    print(f"  {c['source']} — {c['section']} — chunk {c['chunk_id']}")

=== Không filter ===
Based on the provided filing, NVIDIA's revenue growth was driven by:

* **Compute & Networking Revenue Growth:**
  * Driven by major platform shifts in accelerated computing and AI.
  * **Data Center computing** revenue grew 59%, driven by demand for the Blackwell computing platform.
  * **Data Center networking** revenue grew 142%, driven by the introduction and continued ramp of NVLink compute fabric for GB200 and GB300 systems, as well as the growth of Ethernet and InfiniBand platforms.

* **Graphics Revenue Growth:**
  * Driven by sales of the Blackwell architecture.

**Source:** 
* NVIDIA CORP 10-K (Filing Date: 2026-02-25), *Item 7. Management's Discussion and Analysis of Financial Condition and Results of Operations*, Chunk ID: 302.

Citations:
  NVDA_10-K_2026-02-25.html — Item 7. Management's Discussion and Analysis of Financial Condition and Results of Operations — chunk 302
  NVDA_10-K_2026-02-25.html — Item 1. Business — chunk 31
  NVDA_10-Q_2026-05-20.

In [24]:
result_filtered_ticker = ask(
    chain,
    "What are the risk factors related to competition?",
    filters={"ticker": "NVDA"},
)
print("=== Filter ticker=NVDA ===")
print(result_filtered_ticker["answer"])
print()
print("Citations (tất cả phải thuộc NVDA):")
for c in result_filtered_ticker["citations"]:
    print(f"  {c['source']} — {c['section']} — chunk {c['chunk_id']}")
assert all("NVDA" in c["source"] for c in result_filtered_ticker["citations"])

=== Filter ticker=NVDA ===
Based on the provided filings, the risk factors related to competition include:

1. **Market Share and Financial Results:** 
   * Target markets remain competitive, and competition may intensify due to expanding and changing product and service offerings, industry standards, customer and market needs, new entrants, and consolidations, which could adversely impact market share and financial results (SOURCE [3], Section: Part II — Item 1A. Risk Factors, chunk_id: 132).

2. **Demand and Revenue Forecasting:**
   * Competing technologies, competitor product releases, announcements, or other actions, as well as increases in demand for competitive products, are factors that have caused or could cause the company to underestimate or overestimate demand, thereby impacting the timing and volume of revenue (SOURCE [1], Section: Part II — Item 1A. Risk Factors, chunk_id: 140).

3. **Competition for Talent and Rising Labor Costs:**
   * The industry features an increasin

In [25]:
result_10q = ask(
    chain,
    "What legal proceedings does NVIDIA disclose in its most recent 10-Q?",
    filters={"filing_type": "10-Q"},
)
print("=== Filter filing_type=10-Q (minh hoạ section đã gắn Part) ===")
print(result_10q["answer"])
print()
print("Citations:")
for c in result_10q["citations"]:
    print(f"  {c['source']} — {c['section']} — chunk {c['chunk_id']}")

=== Filter filing_type=10-Q (minh hoạ section đã gắn Part) ===
Based on the provided filing, NVIDIA discloses the following legal proceedings and regulatory inquiries:

1. ***In re NVIDIA Corporation Consolidated Derivative Litigation*** (and related ***In Re NVIDIA Corporation Securities Litigation***):
   * **Forum & Caption:** A putative derivative lawsuit pending in the United States District Court for the Northern District of California, captioned `4:19-cv-00341-HSG`, initially filed on January 18, 2019 [SOURCE [4], chunk 60].
   * **Status:** The lawsuit was stayed pending the resolution of the plaintiffs' appeal in the *In Re NVIDIA Corporation Securities Litigation* action. It was administratively closed on February 22, 2022, with the court stating it would reopen the case once the appeal is resolved; the case has not yet been reopened [SOURCE [4], chunk 60].
   * **Claims:** The lawsuit asserts claims on behalf of NVIDIA against certain officers and directors for breach of fid

In [26]:
# Minh hoạ: "documents" và "citations" cùng xuất phát từ MỘT lần retrieve
doc_triples = {(d.metadata["source"], d.metadata["section"], d.metadata["chunk_id"]) for d in result_unfiltered["documents"]}
citation_triples = {(c["source"], c["section"], c["chunk_id"]) for c in result_unfiltered["citations"]}

print(f"Số document trả về (chưa loại trùng): {len(result_unfiltered['documents'])}")
print(f"Số citation (đã loại trùng)         : {len(result_unfiltered['citations'])}")
print("Mọi citation đều nằm trong documents :", citation_triples.issubset(doc_triples))

Số document trả về (chưa loại trùng): 5
Số citation (đã loại trùng)         : 5
Mọi citation đều nằm trong documents : True


---
# PHẦN D — Luồng Index từ giao diện (Streamlit sidebar form)

**Kích hoạt bởi:** người dùng bấm nút "Index filing" trong form
"Index a new filing" trên sidebar của `app/streamlit_app.py`.

> ⚠️ **Lưu ý về khả năng chạy trong notebook:** luồng này gắn chặt với vòng
> đời chạy script của Streamlit (`st.cache_resource`, `st.session_state`,
> `st.rerun()`) — các API này **chỉ hoạt động bên trong một Streamlit
> server thật**, không thể gọi có ý nghĩa từ một Jupyter kernel thường. Vì
> vậy phần này trình bày **lý thuyết + đúng đoạn code nguồn thật**, nhưng
> cell code bên dưới **không được thực thi** trong notebook này (khác với
> mọi phần khác — đây là ngoại lệ kỹ thuật duy nhất, không phải lựa chọn).

## Các bên tham gia

```
User (trình duyệt)
   │  điền ticker + filing_type, bấm "Index filing"
   ▼
Streamlit App (app/streamlit_app.py)
   │
   ▼
index_filing()  ← ĐÚNG HÀM đã giới thiệu ở A.13 — không có luồng thứ hai
   │
   ▼ (thành công)
st.cache_resource.clear()  +  st.rerun()
```

## Vì sao cần `st.cache_resource.clear()` + `st.rerun()`

Streamlit cache 4 tài nguyên "nặng" qua `@st.cache_resource`: embedding
model, FAISS vectorstore, `Retriever`, RAG chain — mỗi cái chỉ được tạo một
lần rồi giữ nguyên trong bộ nhớ suốt vòng đời server, **kể cả khi có filing
mới được index** (vì `index_filing()` ghi thẳng ra đĩa, không tự thông báo
cho các đối tượng đang cache trong bộ nhớ). Nếu không `clear()`, vectorstore
đang cache trong bộ nhớ **vẫn là bản CŨ**, filing mới vừa index sẽ vô hình
với người dùng đến khi restart server.

`st.rerun()` chạy lại toàn bộ script từ đầu — lúc này 4 hàm `@st.cache_resource`
bị gọi lại (vì cache vừa bị xoá), tải lại vectorstore **mới** từ đĩa.

## Vì sao lịch sử chat vẫn sống sót

`st.session_state.messages` (nơi lưu lịch sử chat) là một cơ chế lưu trữ
**hoàn toàn độc lập** với `st.cache_resource` — `clear()` chỉ xoá cache tài
nguyên, không đụng đến `session_state`. Guard
`if "messages" not in st.session_state: st.session_state.messages = []`
ở đầu `main()` (của `streamlit_app.py`) thấy khoá `"messages"` **đã tồn
tại** sau `rerun()` nên không khởi tạo lại — lịch sử chat trước đó hiển thị
lại nguyên vẹn.

In [27]:
# ⚠️ CELL NÀY KHÔNG ĐƯỢC THỰC THI — chỉ minh hoạ đúng logic thật lấy từ
# app/streamlit_app.py:_render_indexing_form(), không thể chạy có ý nghĩa
# ngoài một Streamlit server thật (import streamlit, st.form, st.rerun...
# đều cần Streamlit runtime).
#
# import streamlit as st
# from main import index_filing
#
# with st.form("index_filing_form"):
#     ticker_input = st.text_input("Ticker", placeholder="e.g. NVDA")
#     filing_type_input = st.selectbox("Filing type", ["10-K", "10-Q"])
#     submitted = st.form_submit_button("Index filing")
#
# if submitted:
#     ticker = ticker_input.strip().upper()
#     with st.spinner(f"Indexing {ticker} {filing_type_input}..."):
#         result = index_filing(ticker, filing_type_input)   # <-- CHÍNH HÀM A.13
#     st.success(f"doc_id={result['doc_id']} — {result['status']}")
#     st.cache_resource.clear()   # xoá 4 cache: embedding, vectorstore, retriever, chain
#     st.rerun()                  # chạy lại script; st.session_state.messages KHÔNG bị xoá

---
# PHẦN E — Luồng Đánh giá (Evaluation)

**Kích hoạt bởi:** `python -m src.evaluation.metrics --k 5 --save <label>`

## Các bên tham gia

```
User (terminal)
   │
   ▼
src/evaluation/metrics.py (CLI)
   │
   ├──►  src/evaluation/golden_set.json (file — bộ câu hỏi mẫu có đáp án đúng)
   │
   ├──►  Retriever  ──────────►  FAISS Vector Store    ← DÙNG LẠI retriever Phần C,
   │                                                       KHÔNG đụng tới rag_chain/LLM
   │
   ▼
data/eval/{label}.json (file kết quả, để so sánh giữa các lần chạy)
```

**Xác nhận quan trọng:** toàn bộ `src/evaluation/metrics.py` **không hề
import** `src.generation.rag_chain`, không gọi `get_llm()`, không gọi
`ChatGoogleGenerativeAI` — luồng này dừng lại ở `list[Document]` trả về từ
`Retriever.retrieve` (Phần C.2), không có bước sinh câu trả lời nào.

## E.1 — Dataclass `EvalCase` + `load_golden_set(path) -> list[EvalCase]`

**Dataclass `EvalCase`** (`src/evaluation/metrics.py`):

| Field | Kiểu | Ý nghĩa |
|---|---|---|
| `question` | `str` | Câu hỏi mẫu |
| `relevant` | `set[tuple[str, str]]` | Tập các cặp `(doc_id, section)` được coi là "đúng" cho câu hỏi này — **không phải** `chunk_id`, vì `chunk_id` thay đổi mỗi lần re-chunk, còn `(doc_id, section)` ổn định miễn tiêu đề H2 không đổi |
| `filters` | `dict \| None` | Bộ lọc metadata áp dụng khi truy vấn câu hỏi này (giống hệt tham số `filters` ở C.2/C.5) |

**`load_golden_set(path)`**
- Input: `path` — mặc định `src/evaluation/golden_set.json`.
- Output: `list[EvalCase]` — đọc JSON, chuyển `"relevant"` từ list dict
  thành `set[tuple[str,str]]`.
- Tech stack: `json` thuần.

In [28]:
from src.evaluation.metrics import load_golden_set

cases = load_golden_set()
print(f"Số câu hỏi trong golden set: {len(cases)}")
print()
print("2 case đầu tiên:")
for case in cases[:2]:
    print(f"  question : {case.question}")
    print(f"  relevant : {case.relevant}")
    print(f"  filters  : {case.filters}")
    print()

2026-08-21 19:33:48 | INFO     | src.evaluation.metrics | Loaded 18 golden set case(s) from C:\Python\Personal Project\Financial Report Assistant\src\evaluation\golden_set.json


Số câu hỏi trong golden set: 18

2 case đầu tiên:
  question : What are NVIDIA's main business segments and products?
  relevant : {('NVDA_10-K_2026-02-25', 'Item 1. Business')}
  filters  : None

  question : What risk factors does NVIDIA disclose related to competition?
  relevant : {('NVDA_10-K_2026-02-25', 'Item 1A. Risk Factors')}
  filters  : None



## E.2 — 4 hàm metric thuần: `hit_rate_at_k`, `mrr_at_k`, `recall_at_k`, `precision_at_k`

Cả 4 hàm cùng chữ ký `(retrieved: list[tuple], relevant: set[tuple], k: int) -> float`
— **không cần vectorstore/retriever**, chỉ làm việc trên list/set thuần nên
unit-test được độc lập.

| Hàm | Ý nghĩa | Input `relevant` rỗng |
|---|---|---|
| `hit_rate_at_k` | 1.0 nếu **có ít nhất 1** kết quả đúng trong top-k, ngược lại 0.0 | trả 0.0 |
| `mrr_at_k` | `1/rank` của kết quả đúng **đầu tiên** trong top-k (0.0 nếu không có) | trả 0.0 |
| `recall_at_k` | Tỉ lệ phần tử `relevant` được top-k **bao phủ** (loại trùng trước khi đếm) | trả 0.0 |
| `precision_at_k` | Tỉ lệ top-k **thực sự đúng** (đếm cả trùng lặp — khác `recall_at_k`) | trả 0.0 |

**Tech stack:** thuần Python (`set` giao nhau, slicing list) — không phụ
thuộc thư viện ML/AI nào.

In [29]:
from src.evaluation.metrics import hit_rate_at_k, mrr_at_k, recall_at_k, precision_at_k

# Minh hoạ bằng tay, không qua retriever, để thấy rõ công thức
retrieved_demo = [("NVDA_10-K", "Item 1"), ("NVDA_10-K", "Item 7"), ("NVDA_10-K", "Item 1A")]
relevant_demo = {("NVDA_10-K", "Item 7")}

print("retrieved:", retrieved_demo)
print("relevant :", relevant_demo)
print()
print("hit_rate_at_k  :", hit_rate_at_k(retrieved_demo, relevant_demo, k=3))
print("mrr_at_k       :", mrr_at_k(retrieved_demo, relevant_demo, k=3), "(Item 7 đứng ở rank 2 => 1/2)")
print("recall_at_k    :", recall_at_k(retrieved_demo, relevant_demo, k=3))
print("precision_at_k :", precision_at_k(retrieved_demo, relevant_demo, k=3), "(1 đúng / 3 tổng)")

retrieved: [('NVDA_10-K', 'Item 1'), ('NVDA_10-K', 'Item 7'), ('NVDA_10-K', 'Item 1A')]
relevant : {('NVDA_10-K', 'Item 7')}

hit_rate_at_k  : 1.0
mrr_at_k       : 0.5 (Item 7 đứng ở rank 2 => 1/2)
recall_at_k    : 1.0
precision_at_k : 0.3333333333333333 (1 đúng / 3 tổng)


## E.3 — `evaluate_retrieval(retriever: Retriever, cases: list[EvalCase], k: int = 5) -> dict`

**1. Tên hàm + Input**
- `retriever: Retriever` — **dùng lại chính đối tượng** đã tạo ở C.1 (đây
  chính là điểm "chia sẻ retriever với luồng Query" mà không đụng LLM).
- `cases: list[EvalCase]` — từ E.1.
- `k: int = 5` — ngưỡng cắt cho cả 4 metric.

**2. Output**
- `dict` dạng `{"per_case": [...], "aggregate": {...}}` — điểm trung bình
  4 metric trên toàn bộ `cases`.

**3. Chức năng**
- Với mỗi `case`, gọi `retriever.retrieve(case.question, case.filters)` —
  **chính hàm** C.2 — chuyển kết quả thành `list[(doc_id, section)]`, tính
  4 metric (E.2), rồi lấy trung bình cộng trên toàn bộ.

**4. Tech stack**
- Gọi lại `Retriever` (C.1/C.2) + 4 hàm thuần (E.2) — không thêm thư viện
  mới.

In [30]:
from src.evaluation.metrics import evaluate_retrieval

eval_results = evaluate_retrieval(retriever, cases, k=5)
print("Aggregate (trung bình trên", len(cases), "câu hỏi):")
pprint(eval_results["aggregate"])

2026-08-21 19:33:48 | INFO     | src.evaluation.metrics | case="What are NVIDIA's main business segments and products?" hit_rate@k=1.00 mrr@k=1.00 recall@k=1.00 precision@k=0.60


2026-08-21 19:33:48 | INFO     | src.evaluation.metrics | case='What risk factors does NVIDIA disclose related to competition?' hit_rate@k=1.00 mrr@k=1.00 recall@k=1.00 precision@k=0.80


2026-08-21 19:33:48 | INFO     | src.evaluation.metrics | case='Does NVIDIA have any unresolved SEC staff comments?' hit_rate@k=1.00 mrr@k=1.00 recall@k=1.00 precision@k=0.20


2026-08-21 19:33:48 | INFO     | src.evaluation.metrics | case='What cybersecurity risk management processes does NVIDIA have in place?' hit_rate@k=1.00 mrr@k=1.00 recall@k=1.00 precision@k=0.40


2026-08-21 19:33:48 | INFO     | src.evaluation.metrics | case='What properties does NVIDIA own or lease?' hit_rate@k=1.00 mrr@k=1.00 recall@k=1.00 precision@k=0.20


...

2026-08-21 19:33:49 | INFO     | src.evaluation.metrics | case='What legal proceedings does NVIDIA disclose in its most recent 10-Q?' hit_rate@k=0.00 mrr@k=0.00 recall@k=0.00 precision@k=0.00


2026-08-21 19:33:49 | INFO     | src.evaluation.metrics | case="Have there been material changes to NVIDIA's risk factors since the last 10-K, per the 10-Q?" hit_rate@k=1.00 mrr@k=1.00 recall@k=1.00 precision@k=0.60


2026-08-21 19:33:49 | INFO     | src.evaluation.metrics | case="What is covered in Part II Item 1 of NVIDIA's 10-Q?" hit_rate@k=0.00 mrr@k=0.00 recall@k=0.00 precision@k=0.00


2026-08-21 19:33:49 | INFO     | src.evaluation.metrics | case="What is covered in Part I Item 1 of NVIDIA's 10-Q?" hit_rate@k=1.00 mrr@k=0.50 recall@k=1.00 precision@k=0.40


Aggregate (trung bình trên 18 câu hỏi):
{'hit_rate@k': 0.8888888888888888,
 'mrr@k': 0.8240740740740741,
 'precision@k': 0.3888888888888889,
 'recall@k': 0.8888888888888888}


## E.4 — `save_results`, `load_results`, `compare_results` (tiện ích tái lập)

Ba hàm này không phải bước bắt buộc của luồng đánh giá, mà là công cụ để so
sánh **giữa các lần chạy** (vd trước/sau khi đổi `chunk_size`), tránh phải
đọc log bằng mắt:

- `save_results(results: dict, label: str) -> Path` — ghi
  `data/eval/{label}.json`, kèm timestamp + cấu hình (`chunk_size`,
  `chunk_overlap`, `retrieval_k`, `fetch_k`, `embedding_model_name`).
- `load_results(label_or_path) -> dict` — đọc lại file đã lưu.
- `compare_results(baseline: dict, current: dict) -> dict` — so khớp theo
  từng câu hỏi (không chỉ so trung bình — vì trung bình có thể "hoà" dù một
  câu **fixed** và một câu khác **regressed**), phân loại mỗi câu thành
  `"fixed"` (0.0→1.0), `"regressed"` (1.0→0.0), hoặc `"unchanged"`.

**Tech stack:** `json` (đọc/ghi), `datetime` (timestamp ISO 8601 UTC).

In [31]:
from src.evaluation.metrics import save_results, load_results, compare_results

eval_results["k"] = 5
save_path = save_results(eval_results, "notebook_demo")
print("Đã lưu:", save_path)

reloaded = load_results("notebook_demo")
comparison = compare_results(reloaded, reloaded)  # so với chính nó -> mọi delta = 0
print()
print("So sánh với chính nó (kỳ vọng mọi delta = 0):")
pprint(comparison["aggregate_delta"])
print("Fixed:", len(comparison["fixed"]), " Regressed:", len(comparison["regressed"]), " Unchanged:", len(comparison["unchanged"]))

2026-08-21 19:33:49 | INFO     | src.evaluation.metrics | Saved eval results label='notebook_demo' to C:\Python\Personal Project\Financial Report Assistant\data\eval\notebook_demo.json


2026-08-21 19:33:49 | INFO     | src.evaluation.metrics | Loaded eval results from C:\Python\Personal Project\Financial Report Assistant\data\eval\notebook_demo.json


Đã lưu: C:\Python\Personal Project\Financial Report Assistant\data\eval\notebook_demo.json

So sánh với chính nó (kỳ vọng mọi delta = 0):
{'hit_rate@k': 0.0, 'mrr@k': 0.0, 'precision@k': 0.0, 'recall@k': 0.0}
Fixed: 0  Regressed: 0  Unchanged: 18


---
# PHẦN F — Các hàm/thành phần không thuộc luồng nào cụ thể

Đây là các thành phần **cross-cutting** (dùng ở khắp nơi nhưng không tự
điều phối luồng nào) hoặc **code chết** (được định nghĩa nhưng không có
điểm gọi thực tế nào trong toàn bộ `src/`, `main.py`, `app/`).

## F.1 — `get_logger(name: str) -> logging.Logger`

- **Input:** `name: str` — thường là `__name__` của module gọi.
- **Output:** `logging.Logger` đã cấu hình sẵn 1 `StreamHandler`, định dạng
  `"%(asctime)s | %(levelname)-8s | %(name)s | %(message)s"`, gọi lại nhiều
  lần với cùng `name` không bị nhân đôi handler.
- **Chức năng:** thay thế hoàn toàn cho `print()` trong toàn dự án — mọi
  module (từ `sec_loader.py` đến `app/streamlit_app.py`) đều gọi
  `logger = get_logger(__name__)` ở đầu file.
- **Tech stack:** `logging` (thư viện chuẩn Python) — không phụ thuộc ngoài.

## F.2 — `src/utils/config.py` (không phải hàm — là các hằng số cấu hình)

Module này **chỉ chứa hằng số**, không có logic: đường dẫn (`DATA_DIR`,
`RAW_DIR`, `PROCESSED_DIR`, `VECTORSTORE_DIR`, `MANIFEST_PATH`,
`TICKERS_CACHE_PATH`), tham số chunking (`CHUNK_SIZE=1000`,
`CHUNK_OVERLAP=200`), tham số retrieval (`RETRIEVAL_K=5`, `FETCH_K=20`,
`FILTER_POOL_MULTIPLIER=5`), cấu hình Gemini, endpoint SEC, và
`SYSTEM_PROMPT`. Được import bởi hầu hết mọi module khác nhưng **chính nó
không import bất kỳ module `src` nào** — đây là "lá" của đồ thị import.

**Tech stack:** `python-dotenv` (`load_dotenv()`) — đọc file `.env` vào
biến môi trường trước khi các hằng số `os.getenv(...)` được tính.

## F.3 — Code chết: `load_from_url` và `_parse_cik_accession_from_url`

`load_from_url(url, ticker, filing_type, filing_date) -> FilingRef`
(`src/ingestion/sec_loader.py`) được định nghĩa như một điểm vào thủ công/
debug — bỏ qua bước resolve CIK + gọi Submissions API, tải thẳng từ URL đã
biết. Docstring của chính hàm này tự gọi nó là "Debug/manual entry point".

**Không có điểm gọi nào** trong `src/`, `main.py`, `app/streamlit_app.py`,
hay `tests/` gọi đến hàm này — cũng như hàm phụ trợ duy nhất nó dùng,
`_parse_cik_accession_from_url`. Cả hai tồn tại sẵn sàng cho việc gỡ lỗi thủ
công trong tương lai, nhưng hiện tại **không nằm trong bất kỳ luồng chạy
runtime nào** ở trên.

---
# Tổng kết

| Luồng | Đã minh hoạ bằng | Trạng thái |
|---|---|---|
| A — Index 1 filing | MSFT (thủ công từng hàm) + GOOGL (`index_filing()` trọn gói) | ✅ Chạy live |
| B — Force-rebuild | Toàn bộ 4 filing đã index (NVDA×2, MSFT, GOOGL) | ✅ Chạy live |
| C — Query | 3 câu hỏi thật qua Gemini, có/không filter | ✅ Chạy live |
| D — Index từ UI | Đúng code nguồn từ `app/streamlit_app.py` | ⚠️ Chỉ minh hoạ (cần Streamlit runtime thật) |
| E — Evaluation | `golden_set.json` (17 câu hỏi) qua retriever thật | ✅ Chạy live |

Sau khi chạy xong notebook này, `data/vectorstore/` chứa **4 filing** của
**3 công ty** (NVDA 10-K + 10-Q, MSFT 10-K, GOOGL 10-K), và
`data/eval/notebook_demo.json` chứa kết quả đánh giá vừa chạy.